In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import joblib

df = pd.read_parquet('dane_do_trenowania.parquet')
df = df.drop(columns=['op_unique_carrier', 'dep_delay', 
                      'cancelled', 'NY_TMIN_C', 'NY_TMAX_C'])

df['MIA_WND_sin'] = np.sin(np.deg2rad(df['MIA_WND']))
df['MIA_WND_cos'] = np.cos(np.deg2rad(df['MIA_WND']))

df = df.drop(columns=['MIA_WND'])


X = df.drop(columns=['arr_delay'])
y = df['arr_delay']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
modele = {
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=700, 
                            learning_rate=0.05,
                            max_depth=4,
                            random_state=42, 
                            n_jobs=-1)
}

wytrenowane_modele = {}

for nazwa_modelu, model in modele.items():
    model.fit(X_train, y_train)
    predykcje = model.predict(X_test)
    mae = mean_absolute_error(y_test, predykcje)
    print(f"-> {nazwa_modelu} pomylił się średnio o: {mae:.2f} minut.")

    wytrenowane_modele[nazwa_modelu] = model

In [ ]:
nowy_lot = {
    'month': [6],                 
    'day_of_week': [5],           
    'crs_dep_time': [1730],      
    'crs_arr_time': [2045],      
    'NY_PRCP': [0.0],             
    'NY_AWND': [8.0],             
    'NY_TMP_C': [24.0],           
    'MIA_WND': [90.0],            
    'MIA_CIG': [25000.0],         
    'MIA_VIS': [16000.0],         
    'MIA_TMP_C': [28.0]
}


df_nowy_lot = pd.DataFrame(nowy_lot)

df_nowy_lot['MIA_WND_sin'] = np.sin(np.deg2rad(df_nowy_lot['MIA_WND']))
df_nowy_lot['MIA_WND_cos'] = np.cos(np.deg2rad(df_nowy_lot['MIA_WND']))
df_nowy_lot = df_nowy_lot.drop(columns=['MIA_WND'])

df_nowy_lot = df_nowy_lot.reindex(columns=X_train.columns)


for nazwa_modelu, model in wytrenowane_modele.items():
    wynik = model.predict(df_nowy_lot)[0]
    print(f"Model {nazwa_modelu} przewiduje opóźnienie: {wynik:.2f} minut.")

In [ ]:
model_xgb = wytrenowane_modele['XGBoost']

sciezka_modelu = 'model_xgboost.json'
model_xgb.save_model(sciezka_modelu)

print(f"Sukces! Model XGBoost został zapisany jako: {sciezka_modelu}")